# Part 2 – Rainfall API
**Season:** January 1 to May 31, 2023
**Sources:** Wikipedia (capitals), Open-Meteo geocoding and archive APIs
**Output:** `datos/lluvias_por_departamento.csv`

In [2]:
import io
import time
import unicodedata
import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0"}
SEASON_START = "2023-01-01"
SEASON_END = "2023-05-31"
API_PAUSE = 1  # seconds between API calls

# Same list as Part 1: no accents, so Part 3 can join both tables
DEPARTMENTS = [
    "Amazonas", "Ancash", "Apurimac", "Arequipa", "Ayacucho",
    "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huanuco",
    "Ica", "Junin", "La Libertad", "Lambayeque", "Lima", "Loreto",
    "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
    "San Martin", "Tacna", "Tumbes", "Ucayali"
]

def strip_accents(text):
    """Remove accent marks so 'Áncash' becomes 'Ancash' (same as Part 1)."""
    decomposed = unicodedata.normalize("NFD", str(text))
    return "".join(c for c in decomposed if unicodedata.category(c) != "Mn")

print("Ready:", len(DEPARTMENTS), "departments")

Ready: 25 departments


## 1. Departments and capitals from Wikipedia

In [6]:
url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"
response = requests.get(url, headers=HEADERS)
print("Status:", response.status_code)   # must be 200

tables = pd.read_html(io.StringIO(response.text))
print("Tables found:", len(tables))
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:6])

Status: 200
Tables found: 6
0 (1, 1) ['Departamentos del Perú (con las provincias de régimen especial)']
1 (24, 11) ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
2 (8, 5) ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
3 (2, 11) ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
4 (6, 4) ['Provincia', 'Capital', 'Creación', 'Desaparición']
5 (1, 2) [0, 1]


In [7]:
departments_table = tables[1][["Departamento", "Capital"]]
special_table = tables[3][["Provincia", "Capital"]]

print(departments_table.to_string())
print()
print(special_table.to_string())

     Departamento            Capital
0        Amazonas        Chachapoyas
1          Áncash             Huaraz
2        Apurímac            Abancay
3        Arequipa           Arequipa
4        Ayacucho           Ayacucho
5       Cajamarca          Cajamarca
6           Cusco              Cusco
7    Huancavelica       Huancavelica
8         Huánuco            Huánuco
9             Ica                Ica
10          Junín           Huancayo
11    La Libertad           Trujillo
12     Lambayeque           Chiclayo
13           Lima  Huacho (de facto)
14         Loreto            Iquitos
15  Madre de Dios   Puerto Maldonado
16       Moquegua           Moquegua
17          Pasco     Cerro de Pasco
18          Piura              Piura
19           Puno               Puno
20     San Martín          Moyobamba
21          Tacna              Tacna
22         Tumbes             Tumbes
23        Ucayali           Pucallpa

        Provincia Capital
0  Callao[nota 6]  Callao
1    Lima[nota 7]    L

In [8]:
# Main table: 24 departments
capitals = departments_table.copy()
capitals.columns = ["departamento", "capital"]

# Callao is not in the main table: take it from the special-regime table (same page)
callao = special_table[special_table["Provincia"].str.startswith("Callao")].copy()
callao.columns = ["departamento", "capital"]
capitals = pd.concat([capitals, callao], ignore_index=True)

# Remove Wikipedia notes like [nota 6] and extra spaces
for col in ["departamento", "capital"]:
    capitals[col] = capitals[col].str.replace(r"\[.*?\]", "", regex=True).str.strip()

# Remove accents from department names, same as Part 1
capitals["departamento"] = capitals["departamento"].apply(strip_accents)

# Lima: "Huacho (de facto)" would not be found by the API; use Lima city
capitals.loc[capitals["departamento"] == "Lima", "capital"] = "Lima"

# Check that we have the 25 expected departments, no more, no less
print("Rows:", len(capitals))
print("Missing:", set(DEPARTMENTS) - set(capitals["departamento"]))
print("Extra:  ", set(capitals["departamento"]) - set(DEPARTMENTS))
if len(capitals) != 25 or set(capitals["departamento"]) != set(DEPARTMENTS):
    raise RuntimeError("The capitals table does not have the 25 expected departments.")

capitals

Rows: 25
Missing: set()
Extra:   set()


,departamento,capital
0,Amazonas,Chachapoyas
1,Ancash,Huaraz
2,Apurimac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huanuco,Huánuco
9,Ica,Ica


### Decisions on the capitals table

- **Callao was missing.** Wikipedia’s main table lists 24 departments. Callao is included on the same page in the table of provinces with a special regime, with Callao as its capital. We used that list, which gives the 25 rows expected in the assignment: the 24 departments plus Callao, consistent with the INEI list.
- **Lima's capital appeared as "Huacho (de facto)".** The API cannot find that text. Huacho is only the seat of the regional government of Lima Provincias. The same page lists Lima as the capital of the Province of Lima, where most of the department’s population lives. We use **Lima city**.
- **Footnotes and accents.** We removed Wikipedia notes like `[nota 6]` and stripped accents from department names (`Áncash` → `Ancash`) so they match the names used in Part 1.